In [1]:
# from google.colab import drive
# drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [2]:
!curl -o "/input.txt" "https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1089k  100 1089k    0     0  2720k      0 --:--:-- --:--:-- --:--:-- 2716k


In [3]:
data_path = "/input.txt"

In [4]:
with open(data_path, 'r') as f:
    text = f.read()

In [5]:
len(text)

1115394

In [6]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

batch_size = 32
block_size = 8
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

In [7]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [9]:
stoi= {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

def encoder(s):
    return [stoi[i] for i in s]
def decoder(nums):
    return ''.join([itos[i] for i in nums])

print("encoded:",encoder("hello there"))
print("decoded:", decoder(encoder("hello there")))

encoded: [46, 43, 50, 50, 53, 1, 58, 46, 43, 56, 43]
decoded: hello there


In [10]:
import torch
data = torch.tensor(encoder(text))
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [11]:
n = int(0.9*len(text))
train_data = data[:n]
val_data = data[n:]
print(len(val_data))

111540


In [12]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [13]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(context, target)

tensor([18]) tensor(47)
tensor([18, 47]) tensor(56)
tensor([18, 47, 56]) tensor(57)
tensor([18, 47, 56, 57]) tensor(58)
tensor([18, 47, 56, 57, 58]) tensor(1)
tensor([18, 47, 56, 57, 58,  1]) tensor(15)
tensor([18, 47, 56, 57, 58,  1, 15]) tensor(47)
tensor([18, 47, 56, 57, 58,  1, 15, 47]) tensor(58)


In [14]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split=="train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x,y = x.to(device), y.to(device)
    return x,y

xb,yb = get_batch("train")
print(xb.shape)
print(xb)
print(yb)

torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [15]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

class BigramModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)

        if targets==None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

model = BigramModel(vocab_size).to(device)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss.item())

idx = torch.zeros((1,1), dtype=torch.long)
out = model.generate(idx, max_new_tokens=100)[0].tolist()
print(''.join(decoder(out)))

torch.Size([32, 65])
4.878634929656982

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [18]:
model = BigramModel(vocab_size)
m = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for iter in range(max_iters):

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.5958, val loss 4.6030
step 300: train loss 3.1672, val loss 3.1977
step 600: train loss 2.7341, val loss 2.7597
step 900: train loss 2.5948, val loss 2.6084
step 1200: train loss 2.5417, val loss 2.5717
step 1500: train loss 2.5216, val loss 2.5437
step 1800: train loss 2.5058, val loss 2.5188
step 2100: train loss 2.4967, val loss 2.5413
step 2400: train loss 2.4760, val loss 2.5054
step 2700: train loss 2.4686, val loss 2.4906


In [19]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decoder(m.generate(context, max_new_tokens=500)[0].tolist()))


Th y r vegl mes t'sitTIOrelllO:
Th tion,
Aney ELLan my alotharid yo iserfes.
Larntcerloe CLAto t per chin meak,
on I WhtUMEnoverr ithes, haty hetyo runsp mFierkeCE-piny mbarent, ayof t g at RILIUKNNer&3je, met u mawaient e n; wivirer'
My s w d ESitueatr whomyokeno tl:
BRILARitagof Ing's gnertsk
GBETuly hisane fe thachindo ittouto thte m ayourn;
Giongut' w ci&Qqu s
tth t iple,
Hins h at avendet; Nupu tthere ho pe, haveraw, I pim3thedonPals lleastinde whifo wion o t inaveafa ds l t, d y det d chu 


In [20]:
# self attention

torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [21]:
xbow = torch.zeros((B,T,C)) # bow -> bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b][:t+1] # (t, C)
        xbow[b][t] = torch.mean(xprev, dim=0)

In [22]:
wei = torch.tril(torch.ones(T,T))
wei  = wei / wei.sum(dim=1, keepdim=True)
xbow2 = wei @ x

In [23]:
# third way
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros(T,T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x

In [24]:
torch.allclose(xbow, xbow3, atol=1e-6)

True